---
title: Categorical Data in pandas
created: '2023-06-11T16:07:04-07:00'
date: '2026-11-24T23:20:01-08:00'
authors:
  - bendu
label: categorical-data-in-pandas
license: CC-BY-4.0
tags:
  - computer science
  - programming
  - Python
  - pandas
  - categorical
  - category
  - series
---

**Things on this page are fragmentary and immature notes/thoughts of the author. Please read with your own judgement!**


## `sort_values` vs `groupby` with Categorical Columns

Both `groupby` and `sort_values` have well-known gotchas on `category` columns, though they bite in different ways:

- **`groupby` is the major performance / OOM footgun** (in pandas < 3.0): with `observed=False`, grouping by multiple categorical columns materializes the **full Cartesian product** of all defined categories—including unobserved combinations—which can easily blow up memory and runtime.
- **`sort_values` is a silent ordering footgun**: it sorts by the **order of `cat.categories`**, not lexicographical/alphabetical order, and works even when `ordered=False`.

In [1]:
import pandas as pd

## 1. `groupby` with Categorical Columns (The Cartesian Product Trap)

When grouping by categorical columns, `observed=False` instructs pandas to include every combination of categories from `cat.categories`, even if a combination has zero rows in the DataFrame.

- **pandas < 2.1**: `observed=False` is the silent default.
- **pandas 2.1 – 2.x**: `observed=False` remains the default, but emits a `FutureWarning` recommending an explicit `observed=` argument.
- **pandas >= 3.0**: `observed=True` is the new default.

If you filter a large DataFrame down to a few rows (or group by two high-cardinality categorical columns with 1,000 categories each), `observed=False` creates up to $1{,}000 \times 1{,}000 = 1{,}000{,}000$ groups, causing severe slowdowns or Out-Of-Memory (OOM) errors.

In [2]:
df = pd.DataFrame(
    {
        "region": pd.Categorical(["us", "eu"], categories=["us", "eu", "apac"]),
        "tier": pd.Categorical(["free", "pro"], categories=["free", "pro", "enterprise"]),
        "revenue": [10, 50],
    }
)

# observed=False produces 3 x 3 = 9 groups (7 of them empty, with sum=0; aggregations like mean/min/max give NaN)
df.groupby(["region", "tier"], observed=False)["revenue"].sum()

region  tier      
us      free          10
        pro            0
        enterprise     0
eu      free           0
        pro           50
        enterprise     0
apac    free           0
        pro            0
        enterprise     0
Name: revenue, dtype: int64

In [3]:
# Always pass observed=True (especially on pandas < 3.0) to avoid the Cartesian product
df.groupby(["region", "tier"], observed=True)["revenue"].sum()

region  tier
us      free    10
eu      pro     50
Name: revenue, dtype: int64

## 2. `sort_values` with Categorical Columns

Unlike `object` or `str` columns, `sort_values()` on a `category` column sorts by the **position in `cat.categories`**, *not* alphabetically or numerically.

In [4]:
s = pd.Series([1, 2, 3, 3, 2, 1], dtype="category")
s.sort_values()

0    1
5    1
1    2
4    2
2    3
3    3
dtype: category
Categories (3, int64): [1, 2, 3]

When `categories` are passed explicitly (e.g., via `pd.Categorical(..., categories=...)` or `CategoricalDtype`), `sort_values()` follows that category order rather than lexicographical order (whereas inferred categories are sorted by default):

In [5]:
s_custom = pd.Series(pd.Categorical(["b", "a", "c"], categories=["c", "b", "a"]))
# Sorts as c, b, a (category order), NOT a, b, c!
s_custom.sort_values()

2    c
0    b
1    a
dtype: category
Categories (3, str): ['c', 'b', 'a']

### `ordered=False` vs `ordered=True`

- By default, `dtype="category"` creates an **unordered** categorical (`ordered=False`). Calling `s_custom.min()`, `s_custom.max()`, or comparing with `<` / `>` raises `TypeError`, **yet `sort_values()` still succeeds** and sorts by `cat.categories`.
- If you want lexicographical sorting instead of category-definition order, convert to string (e.g., via `key=lambda col: col.astype(str)` in `DataFrame.sort_values`) or reorder the categories explicitly with `cat.reorder_categories(...)`.

In [6]:
# Sort lexicographically without mutating the column dtype
df_sort = pd.DataFrame({"grade": s_custom, "val": [1, 2, 3]})
df_sort.sort_values(by="grade", key=lambda col: col.astype(str))

,grade,val
1,a,2
0,b,1
2,c,3


## 3. Other Practical Categorical Pitfalls

1. **Filtering retains unused categories**: Filtering a DataFrame or Series does not drop unused entries from `cat.categories`, which can still inflate `groupby(..., observed=False)` or plotting legends. Call `s.cat.remove_unused_categories()` when needed.
2. **`pd.concat` and `merge` can silently upcast**: Concatenating or merging categorical columns whose `categories` differ silently converts the result back to `object` (or `str` in pandas 3.0), losing memory savings. Use `pandas.api.types.union_categoricals` or align categories beforehand.
3. **Assigning unseen values raises `TypeError`**: Assigning a value not present in `cat.categories` fails with `TypeError`. Add new categories first via `s.cat.add_categories(["new_val"])`.